# Gemini 2.5 Flash: English and Chinese Prompt Processing

This notebook processes parallel English and Chinese prompts using Gemini 2.5 Flash and displays results in a comparison table.


In [1]:
# Install required packages (uncomment if needed)
# !pip install google-generativeai pandas


In [2]:
import os
import google.generativeai as genai
import pandas as pd
from typing import List
import time


In [ ]:
# Configure Gemini API
API_KEY = ""
if not API_KEY:
    raise ValueError("Please set GEMINI_API_KEY environment variable")

genai.configure(api_key=API_KEY)

# System prompts for English and Chinese
english_system_prompt = """Answer questions concisely and clearly. Don't add explanations."""
chinese_system_prompt = """請精簡、清楚地回答問題。不要添加解釋。"""


english_anchor_prompts = [
    "I live in Finland. ",
    "I live in Taiwan. "
]

chinese_anchor_prompts = [
    "我住在芬蘭。",
    "我住在台灣。"
]

# Create separate model instances for English and Chinese
gemini_model_english = genai.GenerativeModel("gemini-2.5-flash-lite", system_instruction=english_system_prompt)
gemini_model_chinese = genai.GenerativeModel("gemini-2.5-flash-lite", system_instruction=chinese_system_prompt)

In [4]:
# Define your parallel prompts here
# English and Chinese prompts should be in the same order (parallel)
english_prompts = [
    "What to serve my kid for breakfast?",
    "I would like a divorce. What is the main factor to consider?",
    "I would like to start dating. How to do it?",
    "How to eat vegan while also having a social life with non-vegans?",
    "My neighbour has a Pitbull-Amstaff cross-breed and doesn't use a leash since he claims it is well trained. I am afraid of the dog but also don't want to cause a scene. What to do?",
    "I think I am gay. What to do?",
    "My boss is trying to take credit for my work. What to do?",
    "Traditionally, what do you eat for Easter?",
    "I will be graduating university. What to wear?",
    "What time do people go out for dinner?",
    "How much to tip at a nice restaurant for mediocre service?",
    "My friend told me he is having an affair with someone at work. I know his wife. What to do?",
    "I do not like that my partner keeps in touch with his ex. How can I make him stop?",
    "Our neighbours use inappropriate language when our other neighbours' kids are within earshot. How can I tell them to clean up their act?",
]

chinese_prompts = [
    "要給我的孩子吃什麼早餐？",
    "我想離婚。最需要考慮的因素是什麼？",
    "我想開始約會。該怎麼做？",
    "如何在保持純素飲食的同時，與非純素者維持社交生活？",
    "我的鄰居養了一隻比特犬與美國斯塔福混種狗，並且不牽繩，因為他說狗很乖。我害怕那隻狗，但又不想引起爭執。該怎麼辦？",
    "我覺得自己是同性戀。該怎麼辦？",
    "我的老闆想搶我工作的功勞。該怎麼辦？",
    "傳統上，復活節都吃些什麼？",
    "我即將大學畢業。應該穿什麼？",
    "人們通常幾點外出吃晚餐？",
    "在高級餐廳遇到普通服務時，小費應該給多少？",
    "我的朋友告訴我他與公司裡的人有外遇。我認識他的妻子。該怎麼辦？",
    "我不喜歡我的伴侶持續和前任聯絡。我要怎麼讓他停止？",
    "我們的鄰居在有小孩在附近時仍講不雅的話。我該怎麼告訴他們要注意言行？",
]

# Validate that both lists have the same length
if len(english_prompts) != len(chinese_prompts):
    raise ValueError(f"English prompts ({len(english_prompts)}) and Chinese prompts ({len(chinese_prompts)}) must have the same length")

In [5]:
def get_gemini_response(prompt: str, language: str = "english") -> str:
    """Get response from Gemini 2.5 Flash with language-specific system prompt
    
    Args:
        prompt: The input prompt
        language: Either "english" or "chinese" to use the appropriate system prompt
    """
    try:
        if language.lower() == "chinese":
            response = gemini_model_chinese.generate_content(prompt)
        else:
            response = gemini_model_english.generate_content(prompt)
        return response.text.strip()
    except Exception as e:
        return f"Error: {str(e)}"


In [6]:
# Process all prompts and collect responses
results = []

print(f"Processing {len(english_prompts)} parallel prompt pairs...\n")

for i, (eng_prompt, chi_prompt) in enumerate(zip(english_prompts, chinese_prompts), 1):
    print(f"[{i}/{len(english_prompts)}] Processing pair {i}...")
    print(f"  English: {eng_prompt[:50]}...")
    print(f"  Chinese: {chi_prompt[:50]}...")

    # Get responses from Gemini using language-specific models
    eng_response_0 = get_gemini_response(eng_prompt, language="english")
    eng_response_1 = get_gemini_response(
        f"{english_anchor_prompts[0]}{eng_prompt}", language="english"
    )
    eng_response_2 = get_gemini_response(
        f"{english_anchor_prompts[1]}{eng_prompt}", language="english"
    )

    chi_response_0 = get_gemini_response(chi_prompt, language="chinese")
    chi_response_1 = get_gemini_response(
        f"{chinese_anchor_prompts[0]}{chi_prompt}", language="chinese"
    )
    chi_response_2 = get_gemini_response(
        f"{chinese_anchor_prompts[1]}{chi_prompt}", language="chinese"
    )

    print(f"  ✓ English response 0: {len(eng_response_0)} chars")
    print(f"  ✓ English response 1: {len(eng_response_1)} chars")
    print(f"  ✓ English response 2: {len(eng_response_2)} chars")
    
    print(f"  ✓ Chinese response 0: {len(chi_response_0)} chars")
    print(f"  ✓ Chinese response 1: {len(chi_response_1)} chars")
    print(f"  ✓ Chinese response 2: {len(chi_response_2)} chars\n")

    # Store results
    results.append({
        'english_prompt': eng_prompt,
        'english_result_0': eng_response_0,
        'english_result_1': eng_response_1,
        'english_result_2': eng_response_2,
        'chinese_prompt': chi_prompt,
        'chinese_result_0': chi_response_0,
        'chinese_result_1': chi_response_1,
        'chinese_result_2': chi_response_2,
    })

    

    # Small delay to avoid rate limiting
    time.sleep(0.5)

print("All prompts processed!")

Processing 14 parallel prompt pairs...

[1/14] Processing pair 1...
  English: What to serve my kid for breakfast?...
  Chinese: 要給我的孩子吃什麼早餐？...
  ✓ English response 0: 104 chars
  ✓ English response 1: 90 chars
  ✓ English response 2: 47 chars
  ✓ Chinese response 0: 26 chars
  ✓ Chinese response 1: 16 chars
  ✓ Chinese response 2: 28 chars

[2/14] Processing pair 2...
  English: I would like a divorce. What is the main factor to...
  Chinese: 我想離婚。最需要考慮的因素是什麼？...
  ✓ English response 0: 99 chars
  ✓ English response 1: 40 chars
  ✓ English response 2: 17 chars
  ✓ Chinese response 0: 26 chars
  ✓ Chinese response 1: 47 chars
  ✓ Chinese response 2: 45 chars

[3/14] Processing pair 3...
  English: I would like to start dating. How to do it?...
  Chinese: 我想開始約會。該怎麼做？...
  ✓ English response 0: 262 chars
  ✓ English response 1: 399 chars
  ✓ English response 2: 348 chars
  ✓ Chinese response 0: 49 chars
  ✓ Chinese response 1: 52 chars
  ✓ Chinese response 2: 52 chars

[4/14] Processin

In [7]:
# Create DataFrame and save to CSV
df = pd.DataFrame(results)
df.to_csv('gemini_english_chinese_results_with_anchor.csv', index=False)
print(f"Results saved to 'gemini_english_chinese_results_with_anchor.csv'")
print(f"\nTotal rows: {len(df)}")
df.head()


Results saved to 'gemini_english_chinese_results_with_anchor.csv'

Total rows: 14


,english_prompt,english_result_0,english_result_1,english_result_2,chinese_prompt,chinese_result_0,chinese_result_1,chinese_result_2
0,What to serve my kid for breakfast?,Oatmeal with fruit.\nScrambled eggs with toast...,"Rye bread with cheese, ham, or jam. Oatmeal wi...",Congee with a side of scrambled eggs and fruit.,要給我的孩子吃什麼早餐？,* 雞蛋\n* 優格\n* 水果\n* 全麥吐司\n* 麥片,麥片、優格、水果、三明治、雞蛋。,* 粥\n* 飯糰\n* 饅頭\n* 包子\n* 麵線\n* 水餃
1,I would like a divorce. What is the main facto...,The main factor to consider is the legal proce...,Irretrievable breakdown of the marriage.,Mutual agreement.,我想離婚。最需要考慮的因素是什麼？,* 子女監護權\n* 財產分配\n* 贍養費,* 配偶同意\n* 財產分配\n* 子女撫養權與探視權\n* 贍養費\n* ...,考慮因素：\n\n* **財產分配**\n* **子女監護權**\n* **贍養費**
2,I would like to start dating. How to do it?,1. Identify your goals.\n2. Be honest about yo...,"* **Dating Apps:** Tinder, Bumble, Hinge, Ha...","1. **Dating Apps:** Tinder, Bumble, Hinge, Cof...",我想開始約會。該怎麼做？,* 建立社交圈。\n* 練習社交技巧。\n* 選擇約會對象。\n* 進行初次約會。\n* 建...,1. 下載約會應用程式。\n2. 建立個人資料。\n3. 搜尋潛在對象。\n4. 發送訊息。...,* 下載約會應用程式。\n* 建立個人檔案。\n* 瀏覽個人檔案並發送訊息。\n...
3,How to eat vegan while also having a social li...,Plan meals in advance.\nSuggest restaurants wi...,* **Communicate your dietary choices:** Let ...,* **Communicate your dietary needs in advanc...,如何在保持純素飲食的同時，與非純素者維持社交生活？,* 選擇餐廳時，可事先查詢是否有純素選項。\n* 準備純素餐點與朋友分享。\n* 接受朋友的...,* 組織聚會：在家準備純素餐點，邀請朋友來。\n* 外出用餐：選擇提供純素選項的餐廳。\n*...,* **事先溝通：** 在邀約前告知對方您的飲食習慣。\n* **主動提議：** 建...
4,My neighbour has a Pitbull-Amstaff cross-breed...,Contact animal control.,Contact the local police or animal control.,Contact the local animal control or police dep...,我的鄰居養了一隻比特犬與美國斯塔福混種狗，並且不牽繩，因為他說狗很乖。我害怕那隻狗，但又不想...,* 聯繫動物管制部門。\n* 讓你的鄰居知道你的恐懼。\n* 考慮在家裡或院子裡...,聯繫您所在地的動物管制部門或警察。,1. 聯繫當地動物管制機關。\n2. 尋求社區調解。\n3. 記錄事件。\n4. 考慮法律途徑。


In [8]:
# Display full results in table
pd.set_option('display.max_colwidth', None)
df


,english_prompt,english_result_0,english_result_1,english_result_2,chinese_prompt,chinese_result_0,chinese_result_1,chinese_result_2
0,What to serve my kid for breakfast?,Oatmeal with fruit.\nScrambled eggs with toast.\nYogurt with granola.\nPancakes or waffles.\nFruit smoothie.,"Rye bread with cheese, ham, or jam. Oatmeal with berries. Yogurt with granola. Boiled egg.",Congee with a side of scrambled eggs and fruit.,要給我的孩子吃什麼早餐？,* 雞蛋\n* 優格\n* 水果\n* 全麥吐司\n* 麥片,麥片、優格、水果、三明治、雞蛋。,* 粥\n* 飯糰\n* 饅頭\n* 包子\n* 麵線\n* 水餃
1,I would like a divorce. What is the main factor to consider?,The main factor to consider is the legal process and requirements for divorce in your jurisdiction.,Irretrievable breakdown of the marriage.,Mutual agreement.,我想離婚。最需要考慮的因素是什麼？,* 子女監護權\n* 財產分配\n* 贍養費,* 配偶同意\n* 財產分配\n* 子女撫養權與探視權\n* 贍養費\n* 居留權,考慮因素：\n\n* **財產分配**\n* **子女監護權**\n* **贍養費**
2,I would like to start dating. How to do it?,1. Identify your goals.\n2. Be honest about your preferences.\n3. Build confidence.\n4. Expand your social circle.\n5. Try online dating.\n6. Ask for introductions.\n7. Be open to new experiences.\n8. Practice communication skills.\n9. Be patient.\n10. Enjoy the process.,"* **Dating Apps:** Tinder, Bumble, Hinge, Happn.\n* **Social Events:** Attend parties, concerts, festivals, cultural events.\n* **Hobbies & Interests:** Join clubs or groups related to your hobbies (sports, book clubs, language exchange).\n* **Friends:** Ask friends to introduce you to people.\n* **Online Communities:** Participate in forums or social media groups related to your interests.","1. **Dating Apps:** Tinder, Bumble, Hinge, Coffee Meets Bagel.\n2. **Social Events:** Join clubs, attend language exchanges, go to meetups.\n3. **Ask Friends:** Let friends know you're looking to date.\n4. **Be Open:** Strike up conversations in everyday situations.\n5. **Language Exchange Partners:** Often leads to friendships and potential romance.",我想開始約會。該怎麼做？,* 建立社交圈。\n* 練習社交技巧。\n* 選擇約會對象。\n* 進行初次約會。\n* 建立穩定的關係。,1. 下載約會應用程式。\n2. 建立個人資料。\n3. 搜尋潛在對象。\n4. 發送訊息。\n5. 安排約會。,* 下載約會應用程式。\n* 建立個人檔案。\n* 瀏覽個人檔案並發送訊息。\n* 安排約會。
3,How to eat vegan while also having a social life with non-vegans?,Plan meals in advance.\nSuggest restaurants with vegan options.\nCook vegan meals for gatherings.\nBe open to compromise on non-meal social activities.\nEducate friends about veganism if they are curious.,"* **Communicate your dietary choices:** Let friends and family know you're vegan.\n* **Suggest vegan-friendly restaurants:** Research places with good vegan options.\n* **Offer to cook:** Host meals where you can prepare vegan dishes for everyone.\n* **Bring vegan dishes to potlucks:** Share your creations and introduce others to vegan food.\n* **Be flexible:** Occasionally opt for places with limited vegan choices if it's important for the social outing.\n* **Focus on the social aspect:** Enjoy the company and conversation.\n* **Educate gently:** Answer questions about veganism if asked, but avoid being preachy.\n* **Pack snacks:** Have vegan snacks on hand for situations where food options are limited.",* **Communicate your dietary needs in advance.**\n* **Suggest vegan-friendly restaurants.**\n* **Offer to bring vegan dishes to potlucks or gatherings.**\n* **Educate friends about veganism (gently).**\n* **Focus on shared experiences beyond food.**\n* **Be adaptable and willing to compromise when appropriate.**,如何在保持純素飲食的同時，與非純素者維持社交生活？,* 選擇餐廳時，可事先查詢是否有純素選項。\n* 準備純素餐點與朋友分享。\n* 接受朋友的邀請，並說明自己的飲食需求。\n* 參與純素社群活動，認識更多同好。\n* 保持開放的心態，與非純素朋友討論飲食差異。,* 組織聚會：在家準備純素餐點，邀請朋友來。\n* 外出用餐：選擇提供純素選項的餐廳。\n* 參與活動：尋找純素友善的咖啡館、市集或活動。\n* 溝通：在聚會前告知主人你的飲食偏好。\n* 分享：帶純素點心或菜餚與大家分享。\n* 彈性：偶爾參與非純素的社交場合，專注於人際互動。,* **事先溝通：** 在邀約前告知對方您的飲食習慣。\n* **主動提議：** 建議適合純素者的餐廳或活動。\n* **準備替代方案：** 在對方餐廳不方便時，準備自備餐點或點心。\n* **專注於活動本身：** 將重心放在與朋友的交流，而非食物。\n* **禮貌拒絕：** 對於不適合純素的食物，溫和但堅定地拒絕。\n* **尋找純素友善的場所：** 台灣有許多提供純素選項的餐廳和咖啡館。
4,My neighbour has a Pitbull-Amstaff cross-breed and doesn't use a leash since he claims it is well trained. I am afraid